In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import pandas as pd
import numpy as np
import anndata as ad
import scanpy as sc
import pandas as pd
import os
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)
pd.set_option('display.max_colwidth', None)

# soundlife cohort
96 subjects (donors), 49 young and 47 old
each subject has multiple samples (~10) -> what these samples stand for?


<img src="downloads/image1.png" width="50%">
<img src="downloads/image2.png" width="50%">

In [ ]:
adata = ad.read_h5ad('downloads/SoundLife_OlderAdult_Female_CMVneg.h5ad.2')
obs = adata.obs

In [31]:
obs[obs['subject.subjectGuid']=='BR2036'][['sample.visitName', 'subject.bmi', 'subject.ageAtFirstDraw', 'sample.subjectAgeAtDraw']].drop_duplicates(keep='first').sort_values('sample.visitName')

,sample.visitName,subject.bmi,subject.ageAtFirstDraw,sample.subjectAgeAtDraw
barcodes,,,,
5ec5d62223db11ec8fb6d64cd9da18b7,Flu Year 1 Day 0,23.0,56,56
b255fe9ae8ba11eb951c4a933a1ddc30,Flu Year 1 Day 7,23.0,56,56
2c1bccbcbe4211eb98a80ea06e750864,Flu Year 1 Day 90,23.0,56,57
15e1ac26965211ecbf329a5ba518ef1a,Flu Year 2 Day 0,23.0,56,57
15dc2cf6965211ecbf329a5ba518ef1a,Flu Year 2 Day 7,23.0,56,57
a3d43e8cd52a11edb701e233a95dd86a,Flu Year 2 Day 90,23.0,56,58
3fd704146ffd11ee833f960f38b040be,Flu Year 3 Stand-Alone,23.0,56,58
5bb45fd620a911ee8fef66c3e3d85ce1,Immune Variation Day 0,23.0,56,57
8fde5dc8c09011eb909cd6ff17e70a6b,Immune Variation Day 7,23.0,56,57


In [5]:
!ls -lt /vol/projects/jnourisa/datasets/bulk/parsebioscience_bulk.h5ad

-rw-r--r-- 1 jnourisa clusers 824850503 Nov 27 11:33 /vol/projects/jnourisa/datasets/bulk/parsebioscience_bulk.h5ad


In [7]:
adata = ad.read_h5ad('/vol/projects/jnourisa/datasets/bulk/parsebioscience_bulk.h5ad', backed='r')
adata

AnnData object with n_obs × n_vars = 5760 × 34285 backed at '/vol/projects/jnourisa/datasets/bulk/parsebioscience_bulk.h5ad'
    obs: 'sum_by', 'age', 'cell_type', 'is_control', 'perturbation_type', 'well', 'donor_id', 'perturbation', 'cell_count'
    uns: 'data_reference', 'data_url', 'dataset_description', 'dataset_id', 'dataset_name', 'dataset_organism', 'dataset_summary', 'normalization_id'
    layers: 'lognorm'

In [21]:
adata.obs[['cell_type', 'perturbation', 'donor_id', 'well']].drop_duplicates()

,cell_type,perturbation,donor_id,well
0,B,4-1BBL,Donor10,H5
1,B,4-1BBL,Donor11,H5
2,B,4-1BBL,Donor12,H5
3,B,4-1BBL,Donor1,H5
4,B,4-1BBL,Donor2,H5
...,...,...,...,...
12621,NK,VEGF,Donor4,F1
12622,NK,VEGF,Donor5,F1
12623,NK,VEGF,Donor6,F1
12624,NK,VEGF,Donor7,F1


In [17]:
adata[(adata.obs['is_control'])&(adata.obs['cell_type']=='CD4T')&(adata.obs['donor_id']=='Donor10')].obs

,sum_by,cell_type,perturbation,group,well,cell_type_minor,donor_id,is_control,perturbation_type,cell_count,age
5542,_CD4 Memory_PBS_Donor10_H10,CD4T,PBS,CD4 Memory_PBS_Donor10_H10,H10,CD4 Memory,Donor10,True,cytokine,1088,42
5543,_CD4 Memory_PBS_Donor10_H11,CD4T,PBS,CD4 Memory_PBS_Donor10_H11,H11,CD4 Memory,Donor10,True,cytokine,1107,42
5544,_CD4 Memory_PBS_Donor10_H12,CD4T,PBS,CD4 Memory_PBS_Donor10_H12,H12,CD4 Memory,Donor10,True,cytokine,840,42
5545,_CD4 Memory_PBS_Donor10_H7,CD4T,PBS,CD4 Memory_PBS_Donor10_H7,H7,CD4 Memory,Donor10,True,cytokine,926,42
5546,_CD4 Memory_PBS_Donor10_H8,CD4T,PBS,CD4 Memory_PBS_Donor10_H8,H8,CD4 Memory,Donor10,True,cytokine,1028,42
5547,_CD4 Memory_PBS_Donor10_H9,CD4T,PBS,CD4 Memory_PBS_Donor10_H9,H9,CD4 Memory,Donor10,True,cytokine,903,42
6694,_CD4 Naive_PBS_Donor10_H10,CD4T,PBS,CD4 Naive_PBS_Donor10_H10,H10,CD4 Naive,Donor10,True,cytokine,552,42
6695,_CD4 Naive_PBS_Donor10_H11,CD4T,PBS,CD4 Naive_PBS_Donor10_H11,H11,CD4 Naive,Donor10,True,cytokine,639,42
6696,_CD4 Naive_PBS_Donor10_H12,CD4T,PBS,CD4 Naive_PBS_Donor10_H12,H12,CD4 Naive,Donor10,True,cytokine,519,42
6697,_CD4 Naive_PBS_Donor10_H7,CD4T,PBS,CD4 Naive_PBS_Donor10_H7,H7,CD4 Naive,Donor10,True,cytokine,792,42


In [10]:
# Create age mapping from the donor information
donor_age_map = {
    'Donor1': 75,
    'Donor2': 34,
    'Donor3': 68,
    'Donor4': 59,
    'Donor5': 41,
    'Donor6': 38,
    'Donor7': 45,
    'Donor8': 52,
    'Donor9': 38,
    'Donor10': 42,
    'Donor11': 46,
    'Donor12': 36
}

# Map the age to obs based on donor_id
adata.obs['age'] = adata.obs['donor_id'].map(donor_age_map)
adata.obs

,sum_by,cell_type,perturbation,group,well,cell_type_minor,donor_id,is_control,perturbation_type,cell_count,age
0,_B Intermediate/Memory_4-1BBL_Donor10_H5,B,4-1BBL,B Intermediate/Memory_4-1BBL_Donor10_H5,H5,B Intermediate/Memory,Donor10,False,cytokine,176,42
1,_B Intermediate/Memory_4-1BBL_Donor11_H5,B,4-1BBL,B Intermediate/Memory_4-1BBL_Donor11_H5,H5,B Intermediate/Memory,Donor11,False,cytokine,213,46
2,_B Intermediate/Memory_4-1BBL_Donor12_H5,B,4-1BBL,B Intermediate/Memory_4-1BBL_Donor12_H5,H5,B Intermediate/Memory,Donor12,False,cytokine,247,36
3,_B Intermediate/Memory_4-1BBL_Donor1_H5,B,4-1BBL,B Intermediate/Memory_4-1BBL_Donor1_H5,H5,B Intermediate/Memory,Donor1,False,cytokine,294,75
4,_B Intermediate/Memory_4-1BBL_Donor2_H5,B,4-1BBL,B Intermediate/Memory_4-1BBL_Donor2_H5,H5,B Intermediate/Memory,Donor2,False,cytokine,469,34
...,...,...,...,...,...,...,...,...,...,...,...
17126,_Treg_VEGF_Donor5_F1,CD4T,VEGF,Treg_VEGF_Donor5_F1,F1,Treg,Donor5,False,cytokine,123,41
17127,_Treg_VEGF_Donor6_F1,CD4T,VEGF,Treg_VEGF_Donor6_F1,F1,Treg,Donor6,False,cytokine,281,38
17128,_Treg_VEGF_Donor7_F1,CD4T,VEGF,Treg_VEGF_Donor7_F1,F1,Treg,Donor7,False,cytokine,266,45
17129,_Treg_VEGF_Donor8_F1,CD4T,VEGF,Treg_VEGF_Donor8_F1,F1,Treg,Donor8,False,cytokine,86,52


In [11]:
ad.read_h5ad('/vol/projects/CIIM/perturbation_data/Parse_10M_PBMC_cytokines.h5ad', backed='r').obs

,sample,species,gene_count,tscp_count,mread_count,bc1_wind,bc2_wind,bc3_wind,bc1_well,bc2_well,bc3_well,log1p_n_genes_by_counts,log1p_total_counts,total_counts_MT,pct_counts_MT,log1p_total_counts_MT,donor,cytokine,treatment,cell_type
89_103_005__s1,Donor10_4-1BBL,hg38,2236,4700,8656,89,103,5,H5,p2.A7,A5,7.712891,8.455530,56.0,1.191490,4.043051,Donor10,4-1BBL,cytokine,CD8 Naive
89_103_083__s1,Donor10_4-1BBL,hg38,2222,4337,8235,89,103,83,H5,p2.A7,G11,7.706613,8.375169,71.0,1.637076,4.276666,Donor10,4-1BBL,cytokine,B Naive
89_103_085__s1,Donor10_4-1BBL,hg38,1690,3079,5870,89,103,85,H5,p2.A7,H1,7.433075,8.032685,197.0,6.398181,5.288267,Donor10,4-1BBL,cytokine,B Intermediate/Memory
89_104_009__s1,Donor10_4-1BBL,hg38,1746,3015,5663,89,104,9,H5,p2.A8,A9,7.465655,8.011686,84.0,2.786070,4.442651,Donor10,4-1BBL,cytokine,CD14 Mono
89_104_025__s1,Donor10_4-1BBL,hg38,3182,6986,13153,89,104,25,H5,p2.A8,C1,8.065579,8.851807,165.0,2.361867,5.111988,Donor10,4-1BBL,cytokine,CD14 Mono
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61_186_093__s144,Donor9_VEGF,hg38,1439,2381,4989,61,186,93,F1,p2.H6,H9,7.272398,7.775696,43.0,1.805964,3.784190,Donor9,VEGF,cytokine,CD4 Memory
61_186_108__s144,Donor9_VEGF,hg38,1962,3457,7093,61,186,108,F1,p2.H6,p2.A12,7.582229,8.148446,85.0,2.458779,4.454347,Donor9,VEGF,cytokine,CD14 Mono
61_186_135__s144,Donor9_VEGF,hg38,1589,2785,5983,61,186,135,F1,p2.H6,p2.D3,7.371489,7.932362,49.0,1.759426,3.912023,Donor9,VEGF,cytokine,CD8 Naive
61_186_157__s144,Donor9_VEGF,hg38,1819,3342,7088,61,186,157,F1,p2.H6,p2.F1,7.506592,8.114624,54.0,1.615799,4.007333,Donor9,VEGF,cytokine,CD8 Naive
